# NavAble accessibility detection: train YOLO11n

Trains the NavAble on-device model from the Roboflow dataset `nav-able/navable-accessibility-detection`
(classes: ramp, elevator, tactile, accessible_entrance, accessible_restroom) and exports it for the Flutter app.

**Before you start:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.
You'll be asked for your Roboflow **private** API key (Roboflow > Settings > API Keys). It is only used
in this session to download the dataset and is not saved in the notebook.

At the end, a zip downloads with:
- `navable_yolo11n.tflite` and `labels.txt`: copy both into the app's `assets/models/` folder
- training charts (mAP, precision/recall, confusion matrix) for your manuscript

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 1. Download the dataset from Roboflow

In [ ]:
from getpass import getpass
from roboflow import Roboflow

DATASET_VERSION = 2  # Roboflow dataset version to train on; raise it after adding photos

rf = Roboflow(api_key=getpass('Roboflow private API key: '))
project = rf.workspace('nav-able').project('navable-accessibility-detection')
dataset = project.version(DATASET_VERSION).download('yolov11')
DATA_YAML = f'{dataset.location}/data.yaml'
print(open(DATA_YAML).read())

## 2. Train

YOLO11n is the smallest YOLO11 model, so it runs fast on phones. `patience` stops early when
the validation score stops improving. Takes about 15 to 30 minutes on a free T4 GPU.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # starts from COCO-pretrained weights
model.train(
    data=DATA_YAML,
    epochs=150,
    patience=40,
    imgsz=640,
    batch=16,
    project='navable',
    name='yolo11n',
    exist_ok=True,
    plots=True,
)

## 3. Evaluate on the test split

These are the numbers to report: mAP50, mAP50-95, precision and recall, overall and per class.

In [ ]:
import glob
import os

# Ask the trainer where it saved; newer Ultralytics puts runs under runs/detect/.
try:
    BEST = str(model.trainer.best)
    TRAIN_DIR = str(model.trainer.save_dir)
except NameError:  # session restarted: use the newest trained weights on disk
    BEST = max(glob.glob('**/weights/best.pt', recursive=True), key=os.path.getmtime)
    TRAIN_DIR = os.path.dirname(os.path.dirname(BEST))
print('Best weights:', BEST)

best = YOLO(BEST)
metrics = best.val(data=DATA_YAML, split='test', plots=True)
TEST_DIR = str(metrics.save_dir)
print(f'mAP50:     {metrics.box.map50:.3f}')
print(f'mAP50-95:  {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall:    {metrics.box.mr:.3f}')
for index, name in best.names.items():
    print(f'  {name:<22} mAP50-95 {metrics.box.maps[index]:.3f}')  # maps = per-class mAP50-95

## 4. Export for the app

Exports a LiteRT (TFLite) model (input 640x640 RGB) and the class list in the same order as the model's outputs.

In [ ]:
import glob
import os
import shutil
from ultralytics import YOLO

# Load the newest trained weights from disk, so this cell works on its own.
weights = glob.glob('**/weights/best.pt', recursive=True)
assert weights, 'No trained model found. Run the Train cell first and let it finish.'
BEST = max(weights, key=os.path.getmtime)
print('Exporting', BEST)
best = YOLO(BEST)
# Ultralytics 8.4+ calls TFLite "LiteRT"; full precision (FP16 isn't offered).
exported = str(best.export(format='litert', imgsz=640))
if not exported.endswith('.tflite'):  # some versions return the export folder
    exported = sorted(glob.glob(f'{exported}/**/*.tflite', recursive=True))[0]
shutil.copy(exported, 'navable_yolo11n.tflite')
with open('labels.txt', 'w') as f:
    f.write('\n'.join(best.names[i] for i in range(len(best.names))))
print(open('labels.txt').read())

## 5. Download the model and charts

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile('navable_model.zip', 'w') as z:
    z.write('navable_yolo11n.tflite')
    z.write('labels.txt')
    # Training charts sit next to the weights; test charts exist if Evaluate ran.
    folders = [('train', os.path.dirname(os.path.dirname(BEST)))]
    if 'TEST_DIR' in globals():
        folders.append(('test', TEST_DIR))
    for label, folder in folders:
        for name in os.listdir(folder):
            if name.endswith(('.png', '.jpg', '.csv')):
                z.write(os.path.join(folder, name), f'charts/{label}/{name}')
files.download('navable_model.zip')